In [1]:
import fiftyone as fo
import os

/home/grimlock/miniconda3/envs/default/lib/python3.13/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [3]:
# 1. Setup paths
images_dir = "./New-Datasets/Youtube-scraped-v3/images/train"
labels_dir = "./New-Datasets/Youtube-scraped-v3/labels/train"

# 2. Defines the Class Map
class_map = {
    "0": "helmet",
    "1": "pagdi",
    "2": "no_helmet"    
}

# 3. Create the dataset
dataset = fo.Dataset(name="Indian_Roads_Safety", overwrite=True)
dataset.add_images_dir(images_dir)

# 4. Match labels to images and apply the Class Map
for sample in dataset:
    image_name = os.path.basename(sample.filepath)
    label_name = os.path.splitext(image_name)[0] + ".txt"
    label_path = os.path.join(labels_dir, label_name)

    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            lines = f.readlines()
        
        detections = []
        for line in lines:
            data = line.split()
            
            # Get the class ID (0, 1, or 2) and find the word
            class_id = data[0]
            # .get() prevents crashes if a weird number like '5' accidentally sneaks into your data
            label_name = class_map.get(class_id, "unknown") 
            
            # Convert GroundingDINO/YOLO [cx, cy, w, h] to FiftyOne [x, y, w, h]
            cx, cy, w, h = map(float, data[1:])
            detections.append(
                fo.Detection(
                    label=label_name, 
                    bounding_box=[cx - w/2, cy - h/2, w, h],
                )
            )
        
        # Save the detections to a field named 'ground_truth'
        sample["ground_truth"] = fo.Detections(detections=detections)
        sample.save()

# 5. Launch the App in your Browser
session = fo.launch_app(dataset, auto=False, port=5151)

print("\nServer is running!")
print("CMD+Click the link below to open your data in your browser:")
print("http://localhost:5151")

 100% |█████████████████| 794/794 [498.9ms elapsed, 0s remaining, 1.6K samples/s]      
Session launched. Run `session.show()` to open the App in a cell output.

Server is running!
CMD+Click the link below to open your data in your browser:
http://localhost:5151


In [4]:
# Point directly to the specific dataset folder you want to inspect
DATASET_PATH = "New-Datasets/Youtube-scraped-v3"

images_dir = os.path.join(DATASET_PATH, "images/train")
labels_dir = os.path.join(DATASET_PATH, "labels/train")

# FiftyOne will map class ID 0 -> "helmet", 1 -> "pagdi", 2 -> "no_helmet"
classes = ["helmet", "pagdi", "no_helmet"]

print(f"Loading dataset from: {DATASET_PATH}...")

# --- 2. Create the FiftyOne Dataset ---
dataset = fo.Dataset(name="YOLO_Visualizer", overwrite=True)

# --- 3. The YAML-Bypass Importer ---
# By explicitly providing 'data_path', 'labels_path', and 'classes', 
# FiftyOne acts exactly like a native YOLO dataloader without needing a .yaml file.
dataset.add_dir(
    dataset_type=fo.types.YOLOv4Dataset,
    data_path=images_dir,
    labels_path=labels_dir,
    classes=classes,
    label_field="ground_truth" # Loads your boxes into the 'ground_truth' field
)

print(f"Successfully loaded {len(dataset)} samples!")

# --- 4. Launch the Browser UI ---
session = fo.launch_app(dataset, auto=False, port=5151)

session.config.sync_models = True

print("\nServer is running!")
print("Click the link below to open your data in your browser:")
print("http://localhost:5151")

Loading dataset from: New-Datasets/Youtube-scraped-v3...
 100% |█████████████████| 794/794 [317.8ms elapsed, 0s remaining, 2.5K samples/s]     
Successfully loaded 794 samples!
Session launched. Run `session.show()` to open the App in a cell output.

Server is running!
Click the link below to open your data in your browser:
http://localhost:5151


In [ ]:
export_dir = "New-Datasets/IRD-250-v2"
yolo_classes = ["helmet", "pagdi", "no_helmet"]

dataset.export(
    export_dir=export_dir,
    dataset_type=fo.types.YOLOv5Dataset,
    label_field="ground_truth",
    classes=yolo_classes,
    split="train", # Automatically structures the /train/ and /val/ subdirectories safely
    overwrite=True
)

 100% |███████████████████| 57/57 [122.8ms elapsed, 0s remaining, 464.3 samples/s]     
